In [1]:
import time, traceback, os, requests, json
import pandas as pd
from openpyxl import load_workbook
from sqlalchemy import inspect, text
from datetime import datetime
from zoneinfo import ZoneInfo

from db_utils import save_dataframe, get_engine
from api_utils import fetch_limtFull
from config import SIENGE_USERNAME, SIENGE_PASSWORD, POSTGRES_SCHEMA


In [2]:
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
db = os.getenv("POSTGRES_DB")
schema = os.getenv("POSTGRES_SCHEMA")
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")

_Anapro Tabelao_

In [ ]:
import requests
import pandas as pd
from io import BytesIO

url = "https://anapro-vendas-views-api.azurewebsites.net/views"
params = {
    "token": "token",
    "nome": "nomeTabela",
    "subscription-key": "subscription"
}
headers = {
    "Authorization": "Authorization"
}

response = requests.get(url, params=params, headers=headers)
df = pd.read_excel(BytesIO(response.content))


In [3]:
df.head() # imprime cabeçalho resumido com 5 linhas
#print(df.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df))
#print(df_1[df_1['phone_main'] != True])
#df['id'].duplicated().any() # verifica se existem duplicidades para coluna

,VIEWULTIMAATUALIZACAO,CLIENTEENDERECO,CLIENTECONJUGENOME,CLIENTECONJUGECPF,CLIENTECONJUGEEMAIL,CLIENTECONJUGEEMAIL2,CLIENTECONJUGETELEFONERESIDENCIAL,CLIENTECONJUGETELEFONECOMERCIAL,CHARLESTESTANDO,CLIENTEENDLATITUDE,...,UNIDADERECURSOSPROPRIOSANTESHABITESE,UNIDADERECURSOSPROPRIOSAPOSHABITESE,UNIDADERECURSOSPROPRIOSTOTAL,OBRAINCORPORADOR,USUARIOVENDAAPELIDO,USUARIOVENDAEMAIL,USUARIOVENDANOME,TABELAVENDANOME,POSSUILEAD,STATUSPAGAMENTOECOMMERCE
0,2026-01-16 22:04:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,x,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NÃO,NaN
1,2026-01-16 22:04:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,x,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NÃO,NaN
2,2026-01-16 22:04:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,x,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NÃO,NaN
3,2026-01-16 22:04:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,x,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NÃO,NaN
4,2026-01-16 22:04:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,x,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NÃO,NaN


_Sales Contracts_

In [104]:
all_data = []
offset = 0
limit = 200

while True:
    url = f"https://api.sienge.com.br/olimpo/public/api/v1/sales-contracts?limit={limit}&offset={offset}"
    response = requests.get(url, auth=(api_user, api_password))
    if response.status_code != 200:
        print("Erro:", response.status_code, response.text)
        break

    dados = response.json()
    results = dados.get("results", [])
    if not results:  # se não vier mais nada, encerra
        break
    all_data.extend(results)
    offset += limit
print(f"Total de registros baixados: {len(all_data)}")
df = pd.DataFrame(all_data)

Erro: 503 
Total de registros baixados: 600


In [ ]:
base_sc = df.copy()
base_sc['creation_date'] = datetime.now(ZoneInfo("America/Sao_Paulo"))

In [48]:
# salesContractCustomers
sc_c = base_sc[['id', 'salesContractCustomers']].explode('salesContractCustomers')
sc_c = sc_c.loc[lambda df: df['salesContractCustomers'].apply(lambda x: isinstance(x, dict))]
sc_c = sc_c.rename(columns={"id": "salesContractsId"})
salesContractCustomers = pd.json_normalize(sc_c['salesContractCustomers'])
if not salesContractCustomers.empty:
    salesContractCustomers['salesContractsId'] = sc_c['salesContractsId'].values
# salesContractUnits
sc_u = base_sc[['id', 'salesContractUnits']].explode('salesContractUnits')
sc_u = sc_u.loc[lambda df: df['salesContractUnits'].apply(lambda x: isinstance(x, dict))]
sc_u = sc_u.rename(columns={"id": "salesContractsId"})
salesContractUnits = pd.json_normalize(sc_u['salesContractUnits'])
if not salesContractUnits.empty:
    salesContractUnits['salesContractsId'] = sc_u['salesContractsId'].values
# salesContractPaymentConditions
sc_pc = base_sc[['id', 'paymentConditions']].explode('paymentConditions')
sc_pc = sc_pc.loc[lambda df: df['paymentConditions'].apply(lambda x: isinstance(x, dict))]
sc_pc = sc_pc.rename(columns={"id": "salesContractsId"})
salesContractPaymentConditions = pd.json_normalize(sc_pc['paymentConditions'])
if not salesContractPaymentConditions.empty:
    salesContractPaymentConditions['salesContractsId'] = sc_pc['salesContractsId'].values
# salesContractLinkedCommissions
sc_lc = base_sc[['id', 'linkedCommissions']].explode('linkedCommissions')
sc_lc = sc_lc.loc[lambda df: df['linkedCommissions'].apply(lambda x: isinstance(x, dict))]
sc_lc = sc_lc.rename(columns={"id": "salesContractsId"})
salesContractLinkedCommissions = pd.json_normalize(sc_lc['linkedCommissions'])
if not salesContractLinkedCommissions.empty:
    salesContractLinkedCommissions['salesContractsId'] = sc_lc['salesContractsId'].values
# salesContractLinks
sc_l = base_sc[['id', 'links']].explode('links')
sc_l = sc_l.loc[lambda df: df['links'].apply(lambda x: isinstance(x, dict))]
sc_l = sc_l.rename(columns={"id": "salesContractsId"})
salesContractLinks = pd.json_normalize(sc_l['links'])
if not salesContractLinks.empty:
    salesContractLinks['salesContractsId'] = sc_l['salesContractsId'].values

In [ ]:
df.head().T # imprime cabeçalho resumido com 5 linhas
#print(df.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df))
#print(df_1[df_1['phone_main'] != True])
#df['id'].duplicated().any() # verifica se existem duplicidades para coluna

_api_bearers, cost-centers, units, checking-accounts, companies and creditors_

In [3]:
all_data = []
offset = 0
limit = 200

while True:
    url = f"https://api.sienge.com.br/olimpo/public/api/v1/checking-accounts?limit={limit}&offset={offset}"
    response = requests.get(url, auth=(api_user, api_password))
    if response.status_code != 200:
        print("Erro:", response.status_code, response.text)
        break

    dados = response.json()
    results = dados.get("results", [])
    if not results:  # se não vier mais nada, encerra
        break
    all_data.extend(results)
    offset += limit
print(f"Total de registros baixados: {len(all_data)}")
df_x = pd.DataFrame(all_data)



Total de registros baixados: 401


In [ ]:
df_1 = df_x.copy()

In [95]:
df = df_1.copy()
df = df.rename(columns={"id": "customersId"})

# addresses
ad = df[['customersId', 'addresses']].explode('addresses')
ad = ad[ad['addresses'].apply(lambda x: isinstance(x, dict))]
ad_norm = pd.json_normalize(ad['addresses'])
#ad_norm = ad_norm.rename(columns={
#    "rel": "linksRel",
#    "href": "linksHref"})
ad_norm['customersId'] = ad['customersId'].values
df = df.merge(ad_norm, on='customersId', how='left')


In [5]:
df.head().T # imprime cabeçalho resumido com 5 linhas
#print(df.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df))
#print(df_1[df_1['phone_main'] != True])
#df_1['id'].duplicated().any() # verifica se existem duplicidades para coluna

,0,1,2,3,4
accountNumber,0000333999,0002021145,0002447377,0003219033,0028101-8
accountName,UNICRED FABIAN,BS2 OLIMPO PARTICIPAÇÕES,BANCO BTG - OLIMPO PARTICIPACOES,BANCO BTG - CONTA CORRENTE OLIMPO PARTIC,BRADESCO OPAR 02 EMPREENDIMENTOS
accountType,"{'id': 'B', 'description': 'Bancária'}","{'id': 'B', 'description': 'Bancária'}","{'id': 'I', 'description': 'Investimento'}","{'id': 'B', 'description': 'Bancária'}","{'id': 'B', 'description': 'Bancária'}"
agencyNumber,003106,000001,000001,000001,000402
bankNumber,136,218,208,208,237
bankName,Unicred do Brasil,Banco BS2,Banco BTG,Banco BTG,Banco Bradesco
companyId,44,1,1,1,30
companyName,FABIAN FERNANDES BRUZON,OLIMPO PARTICIPAÇÕES E EMPREENDIMENTOS IMOBILI...,OLIMPO PARTICIPAÇÕES E EMPREENDIMENTOS IMOBILI...,OLIMPO PARTICIPAÇÕES E EMPREENDIMENTOS IMOBILI...,OPAR 02 EMPREENDIMENTOS IMOBILIARIOS SPE LTDA
accountStatus,ENABLED,ENABLED,ENABLED,ENABLED,ENABLED
messageId,NaN,NaN,NaN,NaN,1.0


In [76]:
df1 = df_x.copy()

In [85]:
df = df1.rename(columns={"id": "creditorsId"})

# address
ad = df[['creditorsId', 'address']].copy()
ad = ad[ad['address'].apply(lambda x: isinstance(x, dict))]
ad_norm = pd.json_normalize(ad['address'])
ad_norm = ad_norm.rename(columns={
    "cityId": "addressCityId",
    "cityName": "addressCityName",
    "streetName": "addressStreetName",
    "number": "addressNumber",
    "complement": "addressComplement",
    "neighborhood": "addressNeighborhood",
    "state": "addressState",
    "zipCode": "addressZipCode"})
ad_norm['creditorsId'] = ad['creditorsId'].values
df = df.merge(ad_norm, on='creditorsId', how='left')

# phones
# phones ddd e number
ph = df[['creditorsId', 'phones']].copy()
# garante lista e extrai apenas dicts válidos
ph['phones'] = ph['phones'].apply(lambda x: x if isinstance(x, list) else [])
ph['phones'] = ph['phones'].apply(lambda lst: [p for p in lst if isinstance(p, dict)])
# remove duplicados por ddd+number
ph['phones'] = ph['phones'].apply(lambda lst: list({(p.get('ddd'), p.get('number')): p for p in lst}.values()))
# limita a 3 telefones
ph['phones'] = ph['phones'].apply(lambda lst: lst[:3])
# cria colunas vazias
for i in range(1, 4):
    df[f'phonesddd{i}'] = None
    df[f'phonesNumber{i}'] = None
# preenche colunas
for idx, row in ph.iterrows():
    phones = row['phones']
    for i, p in enumerate(phones, start=1):
        df.loc[df['creditorsId'] == row['creditorsId'], f'phonesddd{i}'] = p.get('ddd')
        df.loc[df['creditorsId'] == row['creditorsId'], f'phonesNumber{i}'] = p.get('number')
# type e observation → somente do primeiro telefone
df['phonesType'] = ph['phones'].apply(lambda lst: lst[0].get('type') if lst else None)
df['phonesObservation'] = ph['phones'].apply(lambda lst: lst[0].get('observation') if lst else None)

# contacts
ct = df[['creditorsId', 'contacts']].copy()
ct['contacts'] = ct['contacts'].apply(lambda x: x if isinstance(x, list) else [])
ct['contacts'] = ct['contacts'].apply(lambda lst: [c for c in lst if isinstance(c, dict)])
# cria colunas vazias
df['contactsName'] = None
df['contactsddd'] = None
df['contactsNumber'] = None
df['contactsExtension'] = None
df['contactsSkype'] = None
df['contactsMsn'] = None
df['contactsEmail1'] = None
df['contactsEmail2'] = None
# percorre cada creditorId
for idx, row in ct.iterrows():
    creditorId = row['creditorsId']
    contacts = row['contacts']
    if not contacts:
        continue
    # 1º contatos
    first = contacts[0]
    df.loc[df['creditorsId'] == creditorId, 'contactsName'] = first.get('name')
    df.loc[df['creditorsId'] == creditorId, 'contactsddd'] = first.get('ddd')
    df.loc[df['creditorsId'] == creditorId, 'contactsNumber'] = first.get('number')
    df.loc[df['creditorsId'] == creditorId, 'contactsExtension'] = first.get('extension')
    df.loc[df['creditorsId'] == creditorId, 'contactsSkype'] = first.get('skype')
    df.loc[df['creditorsId'] == creditorId, 'contactsMsn'] = first.get('msn')
    # EMAILS → coletar todos, deduplicar, limitar a 2
    emails = []
    for c in contacts:
        if isinstance(c.get('email'), str):
            emails.append(c['email'])
    # remove duplicados mantendo ordem
    emails = list(dict.fromkeys(emails))
    # limita a 2
    emails = emails[:2]
    if len(emails) >= 1:
        df.loc[df['creditorsId'] == creditorId, 'contactsEmail1'] = emails[0]
    if len(emails) >= 2:
        df.loc[df['creditorsId'] == creditorId, 'contactsEmail2'] = emails[1]

# links
lk = df[['creditorsId', 'links']].explode('links')
lk = lk[lk['links'].apply(lambda x: isinstance(x, dict))]
lk_norm = pd.json_normalize(lk['links'])
lk_norm = lk_norm.rename(columns={
    "rel": "linksRel",
    "href": "linksHref"})
lk_norm['creditorsId'] = lk['creditorsId'].values
df = df.merge(lk_norm, on='creditorsId', how='left')

df = df.rename(columns={"creditorsId": "id"})
df = df.drop(columns=['address', 'phones', 'emails', 'contacts', 'links', 'otherContactMethods'], errors='ignore')

In [73]:
df = df1.rename(columns={"id": "creditorsId"})

# address
ad = df[['creditorsId', 'address']].copy()
ad = ad[ad['address'].apply(lambda x: isinstance(x, dict))]
ad_norm = pd.json_normalize(ad['address'])
ad_norm = ad_norm.rename(columns={
    "cityId": "addressCityId",
    "cityName": "addressCityName",
    "streetName": "addressStreetName",
    "number": "addressNumber",
    "complement": "addressComplement",
    "neighborhood": "addressNeighborhood",
    "state": "addressState",
    "zipCode": "addressZipCode"})
ad_norm['creditorsId'] = ad['creditorsId'].values
df_ad = df.merge(ad_norm, on='creditorsId', how='left')

# phones
ph = df[['creditorsId', 'phones']].explode('phones')
ph = ph[ph['phones'].apply(lambda x: isinstance(x, dict))]
ph_norm = pd.json_normalize(ph['phones'])
ph_norm = ph_norm.rename(columns={
    "ddd": "phonesddd",
    "number": "phonesNumber",
    "main": "phonesMain",
    "type": "phonesType",
    "extension": "phonesExtension",
    "observation": "phonesObservation"})
ph_norm['creditorsId'] = ph['creditorsId'].values
ph_norm = ph_norm.drop_duplicates(subset=['creditorsId', 'phonesddd', 'phonesNumber'])
df_ph = df_ad.merge(ph_norm, on='creditorsId', how='left')

# contacts
ct = df[['creditorsId', 'contacts']].explode('contacts')
ct = ct[ct['contacts'].apply(lambda x: isinstance(x, dict))]
ct_norm = pd.json_normalize(ct['contacts'])
ct_norm = ct_norm.rename(columns={
    "name": "contactsName",
    "ddd": "contactsddd",
    "number": "contactsNumber",
    "extension": "contactsExtension",
    "email": "contactsEmail",
    "skype": "contactsSkype",
    "msn": "contactsMsn"})
ct_norm['creditorsId'] = ct['creditorsId'].values
df_ct = df_ad.merge(ct_norm, on='creditorsId', how='left')

# links
lk = df[['creditorsId', 'links']].explode('links')
lk = lk[lk['links'].apply(lambda x: isinstance(x, dict))]
lk_norm = pd.json_normalize(lk['links'])
lk_norm = lk_norm.rename(columns={
    "rel": "linksRel",
    "href": "linksHref"})
lk_norm['creditorsId'] = lk['creditorsId'].values
df_lk = df_ct.merge(lk_norm, on='creditorsId', how='left')

#df = df.rename(columns={"creditorsId": "id"})

In [ ]:
df = df1.rename(columns={"id": "creditorsId"})

# -------------------------
# PHONES (compactar em colunas)
# -------------------------

ph = df[['creditorsId', 'phones']].copy()
# garante lista e extrai apenas dicts válidos
ph['phones'] = ph['phones'].apply(lambda x: x if isinstance(x, list) else [])
ph['phones'] = ph['phones'].apply(lambda lst: [p for p in lst if isinstance(p, dict)])
# remove duplicados por ddd+number
ph['phones'] = ph['phones'].apply(lambda lst: list({(p.get('ddd'), p.get('number')): p for p in lst}.values()))
# limita a 3 telefones
ph['phones'] = ph['phones'].apply(lambda lst: lst[:3])
# cria colunas vazias
for i in range(1, 4):
    df[f'phonesddd{i}'] = None
    df[f'phonesNumber{i}'] = None
# preenche colunas
for idx, row in ph.iterrows():
    phones = row['phones']
    for i, p in enumerate(phones, start=1):
        df.loc[df['creditorsId'] == row['creditorsId'], f'phonesddd{i}'] = p.get('ddd')
        df.loc[df['creditorsId'] == row['creditorsId'], f'phonesNumber{i}'] = p.get('number')
# type e observation → somente do primeiro telefone
df['phonesType'] = ph['phones'].apply(lambda lst: lst[0].get('type') if lst else None)
df['phonesObservation'] = ph['phones'].apply(lambda lst: lst[0].get('observation') if lst else None)

In [ ]:
em = df[['creditorsId', 'emails']].copy()
# garante lista, extrai apenas dicts válidos e limita a 3 telefones
em['emails'] = em['emails'].apply(lambda x: x if isinstance(x, list) else [])
em['emails'] = em['emails'].apply(lambda lst: [e for e in lst if isinstance(e, str)])
em['emails'] = em['emails'].apply(lambda lst: lst[:2])
# cria colunas
df['email1'] = None
df['email2'] = None
# preenche
for idx, row in em.iterrows():
    emails = row['emails']
    if len(emails) >= 1:
        df.loc[df['creditorsId'] == row['creditorsId'], 'email1'] = emails[0]
    if len(emails) >= 2:
        df.loc[df['creditorsId'] == row['creditorsId'], 'email2'] = emails[1]

_BU_units_


In [46]:
excel_file = "C:/Users/Moacir Faria/Olimpo Participacoes/OLP - CONTROLE/ANALISES/bancos_dados/Logs.xlsx"
sheet_name = "billId_documentNumber"
table_name = "tb_BU"

In [47]:
wb = load_workbook(excel_file, data_only=True)
ws = wb[sheet_name]

table = ws.tables[table_name]
ref = table.ref
cols_range = ref.split(":")
col_start = ''.join([c for c in cols_range[0] if c.isalpha()])
col_end   = ''.join([c for c in cols_range[1] if c.isalpha()])
usecols = f"{col_start}:{col_end}"
start_row = int(''.join([c for c in cols_range[0] if c.isdigit()]))

df = pd.read_excel(excel_file, sheet_name=sheet_name, usecols=usecols, header=0, skiprows=start_row-1)
df = df.dropna(subset=["bill_doc_number"])

In [50]:
engine = get_engine()
nome = table_name
df_on = df

In [43]:
with engine.begin() as conn:
    inspector = inspect(conn)
    if not inspector.has_table(nome):
        df_on.head(0).to_sql(nome, conn, if_exists='replace', index=False)
    conn.execute(text(f'TRUNCATE TABLE "{nome}" RESTART IDENTITY CASCADE'))
    df_on.to_sql(nome, conn, if_exists='append', index=False)


In [51]:
#print(len(df))
df_on[df_on["billId"]==10286]


,bill_doc_number,billId,documentIdentificationId,documentNumber,businessUnit,businessUnitInvoice,idDocument
9552,10286 / TREQ.12122025_2,10286.0,TREQ,12122025_2,Inter cia,Inter cia,Inter cia


_Baixar outcome usa a mesma base de income_

In [3]:
url = "https://api.sienge.com.br/olimpo/public/api/bulk-data/v1/income?startDate=2014-01-01&endDate=2065-01-01&selectionType=D"
response = requests.get(url, auth=(api_user, api_password))

In [ ]:
if response.status_code == 200:
    dados = response.json()
    # caminho completo do arquivo
    caminho = r"C:\Users\Moacir Faria\Olimpo Participacoes\OLP - CONTROLE\BASES TRABALHADAS\JSON\outcome.json"
    # grava o JSON formatado
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False)                     #, indent=4)        # sem indent
    print("Arquivo salvo com sucesso em:", caminho)
else:
    print("Erro ao acessar API:", response.status_code, response.text)


In [4]:
dados = response.json()
df = pd.DataFrame(dados)
df_2 = pd.json_normalize(dados['data'])

In [5]:
base_IN = pd.json_normalize(df['data']).copy()
base_IN['creation_date'] = datetime.now(ZoneInfo("America/Sao_Paulo"))
base_IN.insert(0, 'id_base_in', range(1,  len(base_IN) + 1))
# Higienização
for c in ["documentIdentificationId", "mainUnit", "documentNumber"]:
    if c in base_IN.columns:
        base_IN[c] = base_IN[c].astype(str).str.strip()
base_IN = base_IN.rename(
    columns={
        "paymentTerm.id": "paymentTermid",
        "paymentTerm.descrition": "paymentTermdescrition"})

In [7]:
in_r = base_IN[['id_base_in', 'receipts']].explode('receipts')
in_r = in_r.loc[lambda df: df['receipts'].apply(lambda x: isinstance(x, dict))]
receipts = pd.json_normalize(in_r['receipts'])
if not receipts.empty:
    receipts['id_base_in'] = in_r['id_base_in'].values
    receipts.insert(0, "id_base_cr_r", range(1, len(receipts) + 1))


In [8]:
receipts.head().T
#print(len(base_IN))

,0,1,2,3,4
id_base_cr_r,1,2,3,4,5
operationTypeId,2,9,2,9,2
operationTypeName,Recebimento,Repactuação,Recebimento,Repactuação,Recebimento
grossAmount,556.35,49.91,556.74,49.95,556.74
monetaryCorrectionAmount,0.0,0.0,0.0,0.0,0.0
interestAmount,0.0,0.0,0.0,0.0,0.0
fineAmount,0.0,0.0,0.0,0.0,0.0
discountAmount,0.0,0.0,0.0,0.0,0.0
taxAmount,0.0,0.0,0.0,0.0,0.0
netAmount,556.35,49.91,556.74,49.95,556.74


In [34]:
#df_exp = df.explode("data").reset_index(drop=True)
#df_exp = df_exp.loc[df_exp["data"].apply(lambda x: isinstance(x, dict))]
# Normaliza o conteúdo de 'data' para colunas
base_OC = pd.json_normalize(df["data"]).copy()
base_OC["creation_date"] = datetime.now(ZoneInfo("America/Sao_Paulo"))
#base_OC.insert(0, "id_Base_oc", range(1, len(base_OC) + 1))
# Higienização
#for c in ["documentIdentificationId", "documentNumber"]:
#    if c in base_OC.columns:
#        base_OC[c] = base_OC[c].astype(str).str.strip()


In [ ]:
base_OC.head()

In [ ]:
#print(response.status_code)   # ex.: 200 significa sucesso
#print(response.reason)        # texto do status, ex.: "OK"
#len(response.content) / (1024*1024)


194456
Index(['data'], dtype='object')


_Baixar API sienge por request_


In [179]:
import os
import requests
import pandas as pd
from pandas import json_normalize
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()

True

In [180]:
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
db = os.getenv("POSTGRES_DB")
schema = os.getenv("POSTGRES_SCHEMA")
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")

_Customers_

In [66]:
all_data = []
offset = 0
limit = 200

while True:
    url = f"https://api.sienge.com.br/olimpo/public/api/v1/customers?limit={limit}&offset={offset}"
    response = requests.get(url, auth=(api_user, api_password))
    if response.status_code != 200:
        print("Erro:", response.status_code, response.text)
        break

    dados = response.json()
    results = dados.get("results", [])
    if not results:  # se não vier mais nada, encerra
        break
    all_data.extend(results)
    offset += limit
print(f"Total de registros baixados: {len(all_data)}")
df = pd.DataFrame(all_data)



Total de registros baixados: 3191


In [ ]:
# phones
df_ex = df.explode("phones").reset_index(drop=True)
df_ex = df_ex.drop_duplicates(subset=["id"]).reset_index(drop=True)
phones_df = json_normalize(df_ex["phones"]).add_prefix("phone_")
phones_df = phones_df.loc[df_ex.index].reset_index(drop=True)
df_1 = pd.concat([df_ex, phones_df], axis=1)

# addresses
df_ex = df_1.explode("addresses").reset_index(drop=True)
df_ex = df_ex.drop_duplicates(subset=["id"]).reset_index(drop=True)
addresses_df = json_normalize(df_ex["addresses"])
addresses_df = addresses_df.loc[df_ex.index].reset_index(drop=True)
df_2 = pd.concat([df_ex, addresses_df], axis=1)

# spouse
df_ex = df_2.copy()
spouse = df_ex["spouse"].apply(lambda x: x if isinstance(x, dict) else {})
spouse_df = json_normalize(spouse)[["cpf", "name", "email", "sex", "birthDate", "cellphoneNumber"]].add_prefix("spouse_")
df_3 = pd.concat([df_ex, spouse_df], axis=1)

df_3["familyIncome"] = df_3["familyIncome"].apply(lambda x: ",".join(map(str, x)) if isinstance(x, list) else x)
cols =["id","name","cpf","cnpj","numberIdentityCard","foreigner","personType","sex","nationality","birthDate","profession","civilStatus","matrimonialRegime","email",
       "phone_type","phone_idd","phone_number","phone_note","createdAt","mailingAddress","type","streetName","number","complement","neighborhood","city","state","zipCode",
       "spouse_name","spouse_cpf","spouse_sex","spouse_birthDate","spouse_email","spouse_cellphoneNumber","familyIncome"]
df_final = df_3.filter(items=cols)
print(df_final.to_string())


In [ ]:
df.head().T # imprime cabeçalho resumido com 5 linhas
#print(df_final.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df_final))
#print(df_1[df_1['phone_main'] != True])

_paymentCategory_

In [3]:
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")
url = "https://api.sienge.com.br/olimpo/public/api/v1/payment-categories"
response = requests.get(url, auth=(api_user, api_password))

In [5]:
dados = response.json()
df = pd.DataFrame(dados)


In [ ]:
#df.head() # imprime cabeçalho resumido com 5 linhas
print(df.to_string()) # mostra todos os dados dentro do dataframe
#print(len(df))
#print(df_1[df_1['phone_main'] != True])
#df['id'].duplicated().any() # verifica se existem duplicidades para coluna

In [6]:
if response.status_code == 200:
    dados = response.json()
    if dados:
        df = pd.DataFrame(dados)

        # Conexão com banco
        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")
        df.to_sql("financialCategories", engine, schema=schema, if_exists="replace", index=False)
        print("Dados importados com sucesso!")
    else:
        print("API respondeu, mas não retornou dados.")
else:
    print("Erro ao acessar API:", response.status_code, response.text)

Dados importados com sucesso!


In [7]:
# dicionário id->name para lookup
id_map = dict(zip(df["id"].astype(str), df["name"]))

def expand_row(row):
    codigo = str(row["id"])
    # pega prefixos progressivos que existam na base
    niveis = [codigo[:i] for i in range(1, len(codigo)+1) if codigo[:i] in id_map]
    cols = {}
    for j, n in enumerate(niveis[:-1]):  # exclui o último (que é o próprio R)
        cols[f"id{j+1}"] = n           # id correspondente
        cols[f"fc{j+1}"] = id_map[n]   # nome correspondente
    
    # adiciona colunas originais da linha
    for c in row.index:
        if c == "name":
            cols["financialCategory"] = row[c]
        else: cols[c] = row[c]
    return cols

# aplica apenas para tpConta = 'R'
df_out = pd.DataFrame([expand_row(r) for _, r in df[df["tpConta"]=="R"].iterrows()])

In [ ]:
def save_dataframe(df, table_name, schema):
#    engine = get_engine()
    df.to_sql(table_name, engine, schema=schema, if_exists="replace", index=False)
    print(f"Dados importados com sucesso na tabela {table_name}!")




In [ ]:
df_out.head()

,id1,fc1,id2,fc2,id3,fc3,id,financialCategory,tpConta,flRedutora,flAtiva,flAdiantamento,flImposto
0,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010101,Receita de Venda de Unidades Imobiliárias,R,N,S,N,N
1,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010102,Receita de Venda de Terreno,R,N,S,N,N
2,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010103,Receita de Frações Ideais,R,N,S,N,N
3,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010104,Receita de Unidades Vendidas em Permuta,R,N,S,N,N
4,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010105,Receita de Vendas de imóveis adq de Terceiros,R,N,S,N,N
